In [14]:
import os
import requests
import aiohttp
import asyncio
import nest_asyncio
import pandas as pd
import time
import scrapy
from scrapy_playwright.page import PageMethod
from bs4 import BeautifulSoup
import nest_asyncio
import glob
import numpy as np
import matplotlib.pyplot as plt
from fuzzywuzzy import fuzz, process
import re


### Le Peng Data

In [18]:
adhoc_leads = (
    pd.read_excel('Non_MR_Leads_Le Peng.xlsx')
    .rename(columns=lambda c: c.upper().strip().replace(' ', '_').replace('-', '_'))
)

# Keep rows where at least one contact number is not NaN
cols = [f'CONTACT_NUMBER_{i}' for i in range(1, 9)]
adhoc_leads = adhoc_leads[adhoc_leads[cols].notna().any(axis=1)]

print(adhoc_leads.shape)
adhoc_leads.head(10)


(4230, 19)


,COMPANY_NAME,PIC_NAME,DESIGNATION,CONTACT_NUMBER_1,CONTACT_NUMBER_2,CONTACT_NUMBER_3,CONTACT_NUMBER_4,CONTACT_NUMBER_5,CONTACT_NUMBER_6,CONTACT_NUMBER_7,CONTACT_NUMBER_8,ADDRESS,EMAIL_1,EMAIL_2,EMAIL_3,EMAIL_4,EMAIL_5,EMAIL_6,EMAIL_7
0,TAMJAI SAMGOR MIXIAN,Micky Gan,NaN,65 6513 1787,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,micky.gan@tamjai-inti.com.sg,NaN,NaN,NaN,NaN,NaN,NaN
2,FORINTECH,Ian Li,NaN,65 8132 3868,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ian.li@forintechasia.com,NaN,NaN,NaN,NaN,NaN,NaN
4,THE DENZY COLLECTIVE,Kelvin Toh,NaN,65 9386 4782,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,kel@denzy.com,NaN,NaN,NaN,NaN,NaN,NaN
5,INTERNATIONAL FOODGNOSTIC,Richard Tay,NaN,65 9687 4191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,richard_tay@foodgnostic.com,NaN,NaN,NaN,NaN,NaN,NaN
6,HOSANNA FREIGHT AND LOGISTICS PTE LTD,Yanti,NaN,65 9677 8810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,yanti.tang@hosannafreight.com.sg,NaN,NaN,NaN,NaN,NaN,NaN
7,YAMATO IZAKAYA,Ryan,NaN,65 9788 8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nil123@gmail.com,NaN,NaN,NaN,NaN,NaN,NaN
8,ASIAWORLD DEVELOPMENTS,Genny Chua,NaN,65 9732 5246,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,asiaworld1000@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN
9,EVERY INTL,Andy Kuo,NaN,65 9382 1449,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,andykuo76@gmail.com,NaN,NaN,NaN,NaN,NaN,NaN
10,LÄDERACH CHOCOLATIER,Alexander,NaN,65 9816 1222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,operations@laderach.com.sg,NaN,NaN,NaN,NaN,NaN,NaN
11,SHINAGI,Iwan,NaN,65 9340 3891,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,iwan.golf@yahoo.com,NaN,NaN,NaN,NaN,NaN,NaN


### Getting Acra Data 

In [19]:

folder_path = "Acra_Data"

# Get all CSV file paths inside the folder
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# Read and combine all CSVs
# Using low_memory=False to avoid DtypeWarning for mixed types
df = pd.concat((pd.read_csv(f, low_memory=False) for f in csv_files), ignore_index=True)


df.columns = df.columns.str.upper()


acra_data = df[[
    "UEN",
    "ENTITY_NAME",
    "BUSINESS_CONSTITUTION_DESCRIPTION",
    "ENTITY_TYPE_DESCRIPTION",
    "ENTITY_STATUS_DESCRIPTION",
    "REGISTRATION_INCORPORATION_DATE",
    "PRIMARY_SSIC_CODE",
    "SECONDARY_SSIC_CODE",
    "STREET_NAME",
    "POSTAL_CODE"
]].copy()

# Convert to proper data types
acra_data['UEN'] = acra_data['UEN'].astype('string')
acra_data['ENTITY_NAME'] = acra_data['ENTITY_NAME'].astype('string')
acra_data['BUSINESS_CONSTITUTION_DESCRIPTION'] = acra_data['BUSINESS_CONSTITUTION_DESCRIPTION'].astype('string')
acra_data['ENTITY_TYPE_DESCRIPTION'] = acra_data['ENTITY_TYPE_DESCRIPTION'].astype('string')
acra_data['ENTITY_STATUS_DESCRIPTION'] = acra_data['ENTITY_STATUS_DESCRIPTION'].astype('string')
acra_data['REGISTRATION_INCORPORATION_DATE'] = pd.to_datetime(acra_data['REGISTRATION_INCORPORATION_DATE'], errors='coerce')

# Clean string columns — trim, remove extra spaces, uppercase
for col in [
    'UEN',
    'ENTITY_NAME',
    'BUSINESS_CONSTITUTION_DESCRIPTION',
    'ENTITY_TYPE_DESCRIPTION',
    'ENTITY_STATUS_DESCRIPTION',
    'STREET_NAME',
    'POSTAL_CODE'
]:
    acra_data[col] = (
        acra_data[col]
        .fillna('')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.upper()
    )

# Replace placeholders with NaN for standardization
acra_data.replace(['NA', 'N/A', '-', ''], np.nan, inplace=True)

# Convert registration date to dd-mm-yyyy string (optional)
acra_data['REGISTRATION_INCORPORATION_DATE'] = acra_data['REGISTRATION_INCORPORATION_DATE'].dt.strftime('%d-%m-%Y')

# Filter only live entities (LIVE COMPANY or LIVE)
acra_data = acra_data[
    acra_data['ENTITY_STATUS_DESCRIPTION'].isin(['LIVE COMPANY', 'LIVE'])
].reset_index(drop=True)

# Exclude specific PRIMARY_SSIC_CODE values (supposedly the data would be 600k plus but when we exclude this would lessen)
exclude_codes = [
    46900, 47719, 47749, 47539, 47536, 56123,
    10711, 10712, 10719, 10732, 10733, 93209
]

acra_data = acra_data[~acra_data['PRIMARY_SSIC_CODE'].isin(exclude_codes)].reset_index(drop=True)

In [20]:
acra_data.head(5)

,UEN,ENTITY_NAME,BUSINESS_CONSTITUTION_DESCRIPTION,ENTITY_TYPE_DESCRIPTION,ENTITY_STATUS_DESCRIPTION,REGISTRATION_INCORPORATION_DATE,PRIMARY_SSIC_CODE,SECONDARY_SSIC_CODE,STREET_NAME,POSTAL_CODE
0,00182000A,AIK SENG HENG,PARTNERSHIP,SOLE PROPRIETORSHIP/ PARTNERSHIP,LIVE,07-02-1975,46302,na,FISHERY PORT ROAD,619742
1,00233500W,ASIA STORE,PARTNERSHIP,SOLE PROPRIETORSHIP/ PARTNERSHIP,LIVE,28-10-1974,46411,20234,SIMS AVENUE,387509
2,00733000J,AIK CHE HIONG,PARTNERSHIP,SOLE PROPRIETORSHIP/ PARTNERSHIP,LIVE,02-11-1974,32909,46900,ANG MO KIO INDUSTRIAL PARK 2A,568049
3,00927000X,A WALIMOHAMED BROS,PARTNERSHIP,SOLE PROPRIETORSHIP/ PARTNERSHIP,LIVE,12-11-1974,46411,66126,JELLICOE ROAD,208767
4,01173000E,ANG TECK MOH DEPARTMENT STORE,PARTNERSHIP,SOLE PROPRIETORSHIP/ PARTNERSHIP,LIVE,30-10-1974,47711,47214,WOODLANDS STREET 12,738623


In [21]:
acra_names = acra_data[["ENTITY_NAME"]]

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def preprocess_company_name(name):
    """
    OPTIMIZED preprocessing for company names.
    Removes common suffixes, special characters, and standardizes format.
    """
    if pd.isna(name):
        return ""

    # Convert to string and uppercase
    name = str(name).upper().strip()

    # Remove special characters and punctuation (keep alphanumeric and spaces)
    name = re.sub(r'[^\w\s]', ' ', name)

    # Remove common company suffixes in order of longest to shortest to avoid partial matches
    suffixes = [
        'PRIVATE LIMITED', 'PTE LTD', 'PTE. LTD.', 'PTE LTD.', 'PTE. LTD',
        'PVT LTD', 'LIMITED', 'LTD', 'SINGAPORE', 'S.G.', 'SG', 'PTE',
        'COMPANY', 'CO', 'CORPORATION', 'CORP', 'INC', 'INCORPORATED',
        'LLC', 'LLP', 'PROPRIETARY', 'PROP'
    ]

    for suffix in suffixes:
        # Remove suffix if it appears at the end (with word boundary)
        if name.endswith(' ' + suffix):
            name = name[:-len(suffix)-1].strip()
        elif name == suffix:  # Handle case where name is only the suffix
            name = ""

    # Remove extra whitespace
    name = re.sub(r'\s+', ' ', name).strip()

    return name

# Preprocess both datasets
print("Preprocessing company names with optimized algorithm...")
company_names_clean = adhoc_leads['COMPANY_NAME'].apply(preprocess_company_name)
acra_names_clean = acra_names['ENTITY_NAME'].apply(preprocess_company_name)

print(f"Company names to match: {len(company_names_clean):,}")
print(f"ACRA entities: {len(acra_names_clean):,}")

# OPTIMIZED TF-IDF Vectorizer with enhanced parameters
print("\nInitializing OPTIMIZED TF-IDF vectorizer...")

vectorizer = TfidfVectorizer(
    analyzer='char_wb',      # Character n-grams with word boundaries
    ngram_range=(2, 5),      # Extended to 5-grams for better context (OPTIMIZED)
    lowercase=True,
    max_df=0.90,             # More restrictive - ignore very common terms (OPTIMIZED)
    min_df=1,
    strip_accents='unicode',
    sublinear_tf=True        # Use sublinear term frequency scaling (OPTIMIZED)
)

# Fit the vectorizer on combined corpus for consistent vocabulary
print("Fitting vectorizer on combined corpus...")
all_names = pd.concat([company_names_clean, acra_names_clean], ignore_index=True)
vectorizer.fit(all_names)

# Transform both datasets
print("Transforming company names to TF-IDF vectors...")
tfidf_company = vectorizer.transform(company_names_clean)
tfidf_acra = vectorizer.transform(acra_names_clean)

print(f"\nTF-IDF matrix shape - Company names: {tfidf_company.shape}")
print(f"TF-IDF matrix shape - ACRA entities: {tfidf_acra.shape}")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out()):,} features")

# ============================================================================
# OPTIMIZED MEMORY-EFFICIENT BATCH PROCESSING
# ============================================================================
print("\n" + "="*70)
print("Computing similarities in BATCHES (optimized memory-efficient approach)")
print("This avoids creating a full similarity matrix (~22GB)")
print("="*70)

BATCH_SIZE = 100  # Process 100 companies at a time
PERFECT_THRESHOLD = 1.00  # Perfect matches (100%)
HIGH_QUALITY_THRESHOLD = 0.85  # High-quality matches (85-99%)
MINIMUM_THRESHOLD = 0.00  # Below 85%

perfect_matches = []
high_quality_matches = []
low_quality_matches = []

total_batches = (len(adhoc_leads) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_num in range(total_batches):
    start_idx = batch_num * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(adhoc_leads))

    # Compute similarity ONLY for this batch (saves memory)
    batch_similarities = cosine_similarity(tfidf_company[start_idx:end_idx], tfidf_acra)

    # Process each company in this batch
    for i, company_idx in enumerate(range(start_idx, end_idx)):
        scores = batch_similarities[i]

        # Find best match
        best_acra_idx = np.argmax(scores)
        best_score = scores[best_acra_idx]

        match_data = {
            'COMPANY_NAME': adhoc_leads.iloc[company_idx]['COMPANY_NAME'],
            'ENTITY_NAME': acra_data.iloc[best_acra_idx]['ENTITY_NAME'],
            'UEN': acra_data.iloc[best_acra_idx]['UEN'],
            'SIMILARITY_SCORE': round(best_score, 4),
            'ENTITY_STATUS': acra_data.iloc[best_acra_idx]['ENTITY_STATUS_DESCRIPTION'],
            'PRIMARY_SSIC_CODE': acra_data.iloc[best_acra_idx]['PRIMARY_SSIC_CODE'],
            'REGISTRATION_DATE': acra_data.iloc[best_acra_idx]['REGISTRATION_INCORPORATION_DATE']
        }

        # NEW: Categorize into 3 tiers based on score
        if best_score == PERFECT_THRESHOLD:
            perfect_matches.append(match_data)
        elif best_score >= HIGH_QUALITY_THRESHOLD:
            high_quality_matches.append(match_data)
        else:
            low_quality_matches.append(match_data)

    # Progress update every 10 batches
    if (batch_num + 1) % 10 == 0 or (batch_num + 1) == total_batches:
        progress = (end_idx / len(adhoc_leads)) * 100
        print(f"  Progress: {end_idx:,} / {len(adhoc_leads):,} companies ({progress:.1f}%)")

print("\n" + "="*70)
print("MATCHING COMPLETE!")
print("="*70)

# Create results dataframes with NEW 3-tier structure
perfect_match_df = pd.DataFrame(perfect_matches)
high_quality_df = pd.DataFrame(high_quality_matches)
low_quality_df = pd.DataFrame(low_quality_matches)

# Summary Statistics
print(f"\n{'='*70}")
print("SUMMARY STATISTICS")
print(f"{'='*70}")
print(f"Total companies processed: {len(adhoc_leads):,}")
print(f"\nPerfect Matches (100%):        {len(perfect_match_df):,} companies ({len(perfect_match_df)/len(adhoc_leads)*100:.2f}%)")
print(f"High-Quality Matches (85-99%): {len(high_quality_df):,} companies ({len(high_quality_df)/len(adhoc_leads)*100:.2f}%)")
print(f"Low-Quality Matches (<85%):    {len(low_quality_df):,} companies ({len(low_quality_df)/len(adhoc_leads)*100:.2f}%)")

# Perfect Match Statistics
if len(perfect_match_df) > 0:
    print(f"\n{'='*70}")
    print("PERFECT MATCHES (100%)")
    print(f"{'='*70}")
    print(f"  Total perfect matches: {len(perfect_match_df):,}")
    print(f"  These companies have exact or near-exact matches in ACRA database")

# High-Quality Match Statistics
if len(high_quality_df) > 0:
    print(f"\n{'='*70}")
    print("HIGH-QUALITY MATCHES (85%-99%)")
    print(f"{'='*70}")
    print(f"  Average similarity: {high_quality_df['SIMILARITY_SCORE'].mean():.4f}")
    print(f"  Median similarity:  {high_quality_df['SIMILARITY_SCORE'].median():.4f}")
    print(f"  Min similarity:     {high_quality_df['SIMILARITY_SCORE'].min():.4f}")
    print(f"  Max similarity:     {high_quality_df['SIMILARITY_SCORE'].max():.4f}")

    # Detailed breakdown
    print(f"\n  Quality Breakdown:")
    very_high = len(high_quality_df[(high_quality_df['SIMILARITY_SCORE'] >= 0.95)])
    high = len(high_quality_df[(high_quality_df['SIMILARITY_SCORE'] >= 0.90) & (high_quality_df['SIMILARITY_SCORE'] < 0.95)])
    good = len(high_quality_df[(high_quality_df['SIMILARITY_SCORE'] >= 0.85) & (high_quality_df['SIMILARITY_SCORE'] < 0.90)])

    print(f"    Very High (0.95-0.99): {very_high:,}")
    print(f"    High (0.90-0.94):      {high:,}")
    print(f"    Good (0.85-0.89):      {good:,}")

# Low-Quality Match Statistics
if len(low_quality_df) > 0:
    print(f"\n{'='*70}")
    print("LOW-QUALITY MATCHES (<85%)")
    print(f"{'='*70}")
    print(f"  Average similarity: {low_quality_df['SIMILARITY_SCORE'].mean():.4f}")
    print(f"  Median similarity:  {low_quality_df['SIMILARITY_SCORE'].median():.4f}")
    print(f"  Min similarity:     {low_quality_df['SIMILARITY_SCORE'].min():.4f}")
    print(f"  Max similarity:     {low_quality_df['SIMILARITY_SCORE'].max():.4f}")
    print(f"\n  ⚠ WARNING: These matches require manual verification!")

    # Additional breakdown for low quality
    print(f"\n  Score Distribution:")
    moderate = len(low_quality_df[(low_quality_df['SIMILARITY_SCORE'] >= 0.70)])
    fair = len(low_quality_df[(low_quality_df['SIMILARITY_SCORE'] >= 0.50) & (low_quality_df['SIMILARITY_SCORE'] < 0.70)])
    poor = len(low_quality_df[(low_quality_df['SIMILARITY_SCORE'] < 0.50)])

    print(f"    Moderate (0.70-0.84): {moderate:,}")
    print(f"    Fair (0.50-0.69):     {fair:,}")
    print(f"    Poor (<0.50):         {poor:,}")

print(f"\n{'='*70}")
print("DATAFRAMES CREATED:")
print(f"{'='*70}")
print(f"  perfect_match_df : {len(perfect_match_df):,} rows (100% similarity)")
print(f"  high_quality_df  : {len(high_quality_df):,} rows (85%-99% similarity)")
print(f"  low_quality_df   : {len(low_quality_df):,} rows (<85% similarity)")

print("\n✓ Optimization improvements:")
print("  - Enhanced preprocessing removes more special characters")
print("  - Extended n-grams (2-5) for better contextual matching")
print("  - Sublinear TF scaling reduces bias toward longer names")
print("  - More restrictive max_df filters common uninformative terms")
print("  - 3-tier classification (100%, 85-99%, <85%)")

Preprocessing company names with optimized algorithm...
Company names to match: 4,230
ACRA entities: 537,328

Initializing OPTIMIZED TF-IDF vectorizer...
Fitting vectorizer on combined corpus...
Transforming company names to TF-IDF vectors...

TF-IDF matrix shape - Company names: (4230, 508517)
TF-IDF matrix shape - ACRA entities: (537328, 508517)
Vocabulary size: 508,517 features

Computing similarities in BATCHES (optimized memory-efficient approach)
This avoids creating a full similarity matrix (~22GB)
  Progress: 1,000 / 4,230 companies (23.6%)
  Progress: 2,000 / 4,230 companies (47.3%)
  Progress: 3,000 / 4,230 companies (70.9%)
  Progress: 4,000 / 4,230 companies (94.6%)
  Progress: 4,230 / 4,230 companies (100.0%)

MATCHING COMPLETE!

SUMMARY STATISTICS
Total companies processed: 4,230

Perfect Matches (100%):        122 companies (2.88%)
High-Quality Matches (85-99%): 1,355 companies (32.03%)
Low-Quality Matches (<85%):    2,753 companies (65.08%)

PERFECT MATCHES (100%)
  Tot

In [24]:
print(f"\n{'='*70}")
print("DATAFRAMES CREATED:")
print(f"{'='*70}")
print(f"  perfect_match_df : {len(perfect_match_df):,} rows (100% similarity)")
print(f"  high_quality_df  : {len(high_quality_df):,} rows (85%-99% similarity)")
print(f"  low_quality_df   : {len(low_quality_df):,} rows (<85% similarity)")



DATAFRAMES CREATED:
  perfect_match_df : 122 rows (100% similarity)
  high_quality_df  : 1,355 rows (85%-99% similarity)
  low_quality_df   : 2,753 rows (<85% similarity)


In [26]:
perfect_match_df.head(10)

,COMPANY_NAME,ENTITY_NAME,UEN,SIMILARITY_SCORE,ENTITY_STATUS,PRIMARY_SSIC_CODE,REGISTRATION_DATE
0,ARTANS HOUSE,ARTANS HOUSE LLP,T16LL1881F,1.0,LIVE,85509,05-10-2016
1,IFS CAPITAL,IFS CAPITAL LIMITED,198700827C,1.0,LIVE COMPANY,64201,28-03-1987
2,AAA MOKITA FOOD MANAGEMENT & CATERING PTE. LTD.,AAA MOKITA FOOD MANAGEMENT & CATERING PTE. LTD.,202012747H,1.0,LIVE COMPANY,56200,04-05-2020
3,ABR HOLDINGS LIMITED,ABR HOLDINGS LIMITED,197803023H,1.0,LIVE COMPANY,47219,23-11-1978
4,COBA COBA,COBA COBA,53289006W,1.0,LIVE,56122,23-01-2015
5,DA SHI TANG CATERING PTE. LTD.,DA SHI TANG CATERING PTE. LTD.,201331601D,1.0,LIVE COMPANY,56111,22-11-2013
6,FLOC CAPITAL PTE LTD,FLOC CAPITAL PRIVATE LIMITED,201943309Z,1.0,LIVE COMPANY,56111,23-12-2019
7,FORTUNE COOKIE PROJECTZ PTE. LTD.,FORTUNE COOKIE PROJECTZ PTE. LTD.,202031966M,1.0,LIVE COMPANY,56111,09-10-2020
8,GRAPEVINE CAFE BAR & RESTAURANT LLP,GRAPEVINE CAFE BAR & RESTAURANT LLP,T07LL0769F,1.0,LIVE,56122,01-06-2007
9,HAO LAI WU FOOD ENTERPRISE,HAO LAI WU FOOD ENTERPRISE,53339840A,1.0,LIVE,47213,20-06-2016


In [27]:
high_quality_df.head(10)

,COMPANY_NAME,ENTITY_NAME,UEN,SIMILARITY_SCORE,ENTITY_STATUS,PRIMARY_SSIC_CODE,REGISTRATION_DATE
0,INTERNATIONAL FOODGNOSTIC,INTERNATIONAL FOODGNOSTICS CORPORATION PTE. LTD.,201315915M,0.8505,LIVE COMPANY,10799,12-06-2013
1,HOSANNA FREIGHT AND LOGISTICS PTE LTD,HOSANNA FREIGHT & LOGISTICS PTE. LTD.,201421035D,0.9715,LIVE COMPANY,52292,17-07-2014
2,YAMATO IZAKAYA,YAMATO IZAKAYA (PTE. LTD.),201733803C,0.9936,LIVE COMPANY,56111,23-11-2017
3,ASIAWORLD DEVELOPMENTS,ASIAWORLD DEVELOPMENTS,52926335A,1.0000,LIVE,46308,11-08-2000
4,FOOD JOY,FOOD-JOY PTE. LTD.,200408793M,0.9759,LIVE COMPANY,47102,14-07-2004
5,TIONG HOE SPECIALTY COFFEE,TIONG HOE SPECIALTY COFFEE PTE. LTD.,201404489C,0.9940,LIVE COMPANY,10763,18-02-2014
6,WOK HEY,WOK HEY PTE. LTD.,201617998K,0.9813,LIVE COMPANY,56121,01-07-2016
7,EG INNOVATIONS,EG INNOVATIONS PTE. LTD.,200100990G,0.9853,LIVE COMPANY,58202,15-02-2001
8,WESTLAKE,WESTLAKE 3 PTE. LTD.,200305400H,0.9531,LIVE COMPANY,56111,11-06-2003
9,MR KNEADY,MR. KNEADY'S,53359762E,0.9914,LIVE,56111,02-04-2017


In [28]:
low_quality_df.head(10)

,COMPANY_NAME,ENTITY_NAME,UEN,SIMILARITY_SCORE,ENTITY_STATUS,PRIMARY_SSIC_CODE,REGISTRATION_DATE
0,TAMJAI SAMGOR MIXIAN,SAMGOR SERVICES PTE. LTD.,202508737N,0.5474,LIVE COMPANY,49231,28-02-2025
1,FORINTECH,FORINTECH INTERNATIONAL PTE. LTD.,202239911M,0.7967,LIVE COMPANY,43304,09-11-2022
2,THE DENZY COLLECTIVE,THE S COLLECTIVE PTE. LTD.,202112447H,0.6896,LIVE COMPANY,70205,08-04-2021
3,EVERY INTL,EVERY TECH PTE. LTD.,202208061N,0.5799,LIVE COMPANY,62011,08-03-2022
4,LÄDERACH CHOCOLATIER,GODIVA CHOCOLATIER (ASIA) LIMITED,T01FC6089K,0.5811,LIVE COMPANY,47219,19-06-2001
5,SHINAGI,AVPVI INAGI SG HOLDING PTE. LTD.,202133752D,0.4111,LIVE COMPANY,64202,28-09-2021
6,TOHO SINGAPORE,TOHO GAS SINGAPORE PTE. LTD.,202500123Z,0.7101,LIVE COMPANY,46610,02-01-2025
7,NIVEKEN PTE LTD,NIVEK TRADING,53483243A,0.5172,LIVE,46212,27-03-2024
8,RESTAURANT BRANDS INTERNATIONAL,INTERNATIONAL BRANDS PTE LTD,199801648N,0.7791,LIVE COMPANY,46303,02-04-1998
9,MINTPLUS,PRINTPLUS DESIGN SERVICES,53002562X,0.6255,LIVE,74192,04-09-2003


### Processing low quality data

In [ ]:
# if len(low_quality_df) > 0:
#     print("="*70)
#     print("RE-PROCESSING LOW-QUALITY MATCHES")
#     print("Finding top 3 similar companies for manual review")
#     print("="*70)
    
#     # Extract company names from low_quality_df
#     low_quality_companies = low_quality_df[['COMPANY_NAME']].drop_duplicates()
    
#     print(f"\nProcessing {len(low_quality_companies):,} low-quality companies...")
    
#     # Preprocess low-quality company names
#     low_quality_clean = low_quality_companies['COMPANY_NAME'].apply(preprocess_company_name)
    
#     # Transform to TF-IDF vectors using the existing vectorizer
#     tfidf_low_quality = vectorizer.transform(low_quality_clean)
    
#     # Process in batches to avoid memory issues
#     BATCH_SIZE = 50  # Smaller batch size for top-3 processing
#     TOP_N = 3  # Get top 3 matches
    
#     enhanced_matches = []
#     total_batches = (len(low_quality_companies) + BATCH_SIZE - 1) // BATCH_SIZE
    
#     for batch_num in range(total_batches):
#         start_idx = batch_num * BATCH_SIZE
#         end_idx = min(start_idx + BATCH_SIZE, len(low_quality_companies))
        
#         # Compute similarities for this batch
#         batch_similarities = cosine_similarity(
#             tfidf_low_quality[start_idx:end_idx], 
#             tfidf_acra
#         )
        
#         # Process each company in the batch
#         for i, company_idx in enumerate(range(start_idx, end_idx)):
#             scores = batch_similarities[i]
            
#             # Get top 3 match indices
#             top_3_indices = np.argsort(scores)[::-1][:TOP_N]
            
#             # Build match record
#             match_record = {
#                 'COMPANY_NAME': low_quality_companies.iloc[company_idx]['COMPANY_NAME']
#             }
            
#             # Add top 3 matches
#             for rank, acra_idx in enumerate(top_3_indices, 1):
#                 score = scores[acra_idx]
#                 match_record[f'MATCH_{rank}_ENTITY_NAME'] = acra_data.iloc[acra_idx]['ENTITY_NAME']
#                 match_record[f'MATCH_{rank}_UEN'] = acra_data.iloc[acra_idx]['UEN']
#                 match_record[f'MATCH_{rank}_SIMILARITY'] = round(score, 4)
#                 match_record[f'MATCH_{rank}_STATUS'] = acra_data.iloc[acra_idx]['ENTITY_STATUS_DESCRIPTION']
#                 match_record[f'MATCH_{rank}_SSIC'] = acra_data.iloc[acra_idx]['PRIMARY_SSIC_CODE']
            
#             enhanced_matches.append(match_record)
        
#         # Progress update
#         if (batch_num + 1) % 5 == 0 or (batch_num + 1) == total_batches:
#             progress = (end_idx / len(low_quality_companies)) * 100
#             print(f"  Progress: {end_idx:,} / {len(low_quality_companies):,} companies ({progress:.1f}%)")
    
#     # Create enhanced dataframe
#     low_quality_enhanced_df = pd.DataFrame(enhanced_matches)
    
#     print("\n" + "="*70)
#     print("ENHANCED LOW-QUALITY DATAFRAME CREATED")
#     print("="*70)
#     print(f"  DataFrame: low_quality_enhanced_df")
#     print(f"  Rows: {len(low_quality_enhanced_df):,}")
#     print(f"  Columns: {len(low_quality_enhanced_df.columns)}")
#     print(f"\n  Columns included:")
#     print(f"    - COMPANY_NAME")
#     print(f"    - MATCH_1_ENTITY_NAME, MATCH_1_UEN, MATCH_1_SIMILARITY, MATCH_1_STATUS, MATCH_1_SSIC")
#     print(f"    - MATCH_2_ENTITY_NAME, MATCH_2_UEN, MATCH_2_SIMILARITY, MATCH_2_STATUS, MATCH_2_SSIC")
#     print(f"    - MATCH_3_ENTITY_NAME, MATCH_3_UEN, MATCH_3_SIMILARITY, MATCH_3_STATUS, MATCH_3_SSIC")
    
#     # Show statistics
#     print(f"\n  Top Match Statistics:")
#     print(f"    Average similarity (Match 1): {low_quality_enhanced_df['MATCH_1_SIMILARITY'].mean():.4f}")
#     print(f"    Average similarity (Match 2): {low_quality_enhanced_df['MATCH_2_SIMILARITY'].mean():.4f}")
#     print(f"    Average similarity (Match 3): {low_quality_enhanced_df['MATCH_3_SIMILARITY'].mean():.4f}")
    
#     # Count how many now exceed 80% threshold
#     high_quality_promoted = len(low_quality_enhanced_df[low_quality_enhanced_df['MATCH_1_SIMILARITY'] >= 0.80])
#     if high_quality_promoted > 0:
#         print(f"\n  ⚠ NOTE: {high_quality_promoted} companies now have Match 1 >= 80%")
#         print(f"         Consider reviewing these for promotion to high-quality matches")
    
#     print("\nShowing first 10 enhanced low-quality matches:")
#     low_quality_enhanced_df.head(10)
# else:
#     print("\nNo low-quality matches to enhance (all matches were either high-quality or no match).")
#     low_quality_enhanced_df = pd.DataFrame()

In [ ]:
# low_quality_enhanced_df.tail(50)